# HopSkipJump Attack on All Models

This notebook assumes you already have the following helper functions available
in the environment (import them from your own modules before running this):

- `run_logreg("CSVs/newDataset.csv")`
- `run_neuralnet("CSVs/newDataset.csv")`
- `run_randomforest("CSVs/newDataset.csv")`
- `run_svm("CSVs/newDataset.csv")`
- `run_xgboost("CSVs/newDataset.csv")`

Each function is expected to return a tuple:

```python
(model, X_test, y_test)
```

where:
- `model` is a trained classifier (pipeline or estimator)
- `X_test` is the test feature matrix (pandas DataFrame or numpy array)
- `y_test` is the corresponding true labels

The notebook then applies the HopSkipJump attack from ART to each model and
reports clean vs adversarial accuracy.


In [3]:
# Imports

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score

# Adversarial Robustness Toolbox (ART)
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump

# IMPORTANT:
# Make sure you import your run_* helpers here, for example:
#
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost
#
# or import your wrapper functions that already call those.


import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but StandardScaler was fitted with feature names",
    category=UserWarning,
)



In [4]:
# Set seed for reproducibility

SEED = 42
np.random.seed(SEED)

In [5]:
# Run all models on the same dataset path
# Assumes each function returns (model, X_test, y_test)

DATA_PATH = "CSVs/newDataset.csv"

model_runs = {}

print("Running Logistic Regression...")
logreg_model, logreg_X_test, logreg_y_test = run_logreg(DATA_PATH)
model_runs["LogisticRegression"] = (logreg_model, logreg_X_test, logreg_y_test)

print("Running Neural Net...")
nn_model, nn_X_test, nn_y_test = run_neuralnet(DATA_PATH)
model_runs["NeuralNet"] = (nn_model, nn_X_test, nn_y_test)

print("Running Random Forest...")
rf_model, rf_X_test, rf_y_test = run_randomforest(DATA_PATH)
model_runs["RandomForest"] = (rf_model, rf_X_test, rf_y_test)

print("Running SVM...")
svm_model, svm_X_test, svm_y_test = run_svm(DATA_PATH)
model_runs["SVM"] = (svm_model, svm_X_test, svm_y_test)

print("Running XGBoost...")
xgb_model, xgb_X_test, xgb_y_test = run_xgboost(DATA_PATH)
model_runs["XGBoost"] = (xgb_model, xgb_X_test, xgb_y_test)

print("\nSummary of collected models:")
for name, (model, X_test, y_test) in model_runs.items():
    print(f" - {name}: model={type(model)}, X_test shape={getattr(X_test, 'shape', None)}")



Running Logistic Regression...
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.936)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699


Confusion Matrix:
         Pred 0  Pred 1
True 0    1268      84
True 1      58     289

AUC: 0.936

=== All results and summaries saved successfully ===
Running Neural Net...
Using parameters: {'hidden_layer_sizes': '128,64', 'alpha': 0.0001, 'learning_rate_

C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [13:31:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [6]:
# HopSkipJump attack on each model

import xgboost as xgb
from art.estimators.classification import SklearnClassifier, XGBoostClassifier
from art.attacks.evasion import HopSkipJump
import numpy as np
import pandas as pd

hsj_kwargs = dict(
    max_iter=20,
    max_eval=10000,
    init_eval=100,
    init_size=10,
    targeted=False,
    norm=2,
)

results_hsj = {}

# Limit samples per model (keep or change to match your notebook)
MAX_SAMPLES = 200


def make_art_classifier(model, X, y, clip_values=(0, 1)):
    """
    Wraps the model in the correct ART classifier.
    Uses XGBoostClassifier for XGBoost models, SklearnClassifier otherwise.
    """
    # XGBoost branch
    if isinstance(model, xgb.XGBClassifier):
        print("Wrapping model with ART XGBoostClassifier for HopSkipJump")
        nb_classes = int(len(np.unique(y)))
        nb_features = int(X.shape[1])
        return XGBoostClassifier(
            model=model,
            nb_features=nb_features,
            nb_classes=nb_classes,
            clip_values=clip_values,
        )

    # Default: sklearn-compatible models
    print("Wrapping model with ART SklearnClassifier for HopSkipJump")
    return SklearnClassifier(model=model, clip_values=clip_values)


for name, (model, X_test, y_test) in model_runs.items():
    print("\n=== HopSkipJump attack on model:", name, "===")

    # Convert data to numpy
    if hasattr(X_test, "to_numpy"):
        X_all = X_test.to_numpy().astype(np.float32)
    else:
        X_all = np.asarray(X_test, dtype=np.float32)

    if hasattr(y_test, "to_numpy"):
        y_all = y_test.to_numpy()
    else:
        y_all = np.asarray(y_test)

    # Optional: limit number of samples
    n_total = len(X_all)
    n = min(MAX_SAMPLES, n_total)
    X = X_all[:n]
    y = y_all[:n]

    # Ensure labels are integer-encoded 1D
    if isinstance(y, pd.Series):
        y = y.to_numpy()
    y_int = y.astype(int).reshape(-1)

    n = X.shape[0]
    print(f"Using {n} samples for HopSkipJump on {name}")

    # Build ART classifier (XGBoost or Sklearn)
    art_classifier = make_art_classifier(model, X, y_int, clip_values=(0, 1))

    # Create HopSkipJump instance
    hsj_attack = HopSkipJump(classifier=art_classifier, **hsj_kwargs)

    clean_correct = 0
    adv_correct = 0

    for i in range(n):
        xi = X[i:i+1]
        yi = y_int[i:i+1]

        # Prediction on clean input
        pred_clean = np.argmax(art_classifier.predict(xi), axis=1)

        # Generate adversarial example (HopSkipJump is decision-based)
        x_adv = hsj_attack.generate(x=xi, y=yi)

        # Prediction on adversarial input
        pred_adv = np.argmax(art_classifier.predict(x_adv), axis=1)

        # Compare scalars explicitly
        clean_correct += int(pred_clean[0] == yi[0])
        adv_correct   += int(pred_adv[0]   == yi[0])

    clean_acc = clean_correct / n
    adv_acc = adv_correct / n

    print(f"{name} clean acc (HSJ): {clean_acc:.4f}")
    print(f"{name} adv   acc (HSJ): {adv_acc:.4f}")

    results_hsj[name] = dict(clean_acc=float(clean_acc), adv_acc=float(adv_acc))

hsj_results_df = pd.DataFrame(results_hsj).T
hsj_results_df



=== HopSkipJump attack on model: LogisticRegression ===
Using 200 samples for HopSkipJump on LogisticRegression
Wrapping model with ART SklearnClassifier for HopSkipJump


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 499.74it/s]


LogisticRegression clean acc (HSJ): 0.9100
LogisticRegression adv   acc (HSJ): 0.6650

=== HopSkipJump attack on model: NeuralNet ===
Using 200 samples for HopSkipJump on NeuralNet
Wrapping model with ART SklearnClassifier for HopSkipJump


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 333.38it/s]


NeuralNet clean acc (HSJ): 0.9600
NeuralNet adv   acc (HSJ): 0.6400

=== HopSkipJump attack on model: RandomForest ===
Using 200 samples for HopSkipJump on RandomForest
Wrapping model with ART SklearnClassifier for HopSkipJump


HopSkipJump:   0%|          | 0/1 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ================================================================
#   SKLEARN METRICS SUMMARY FOR CLEAN + HOPSKIPJUMP PERFORMANCE
# ================================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np
import pandas as pd


hsj_summary_rows = []

print("\n\n===================== HOPSKIPJUMP METRICS SUMMARY =====================\n")

for name, (model, X_test, y_test) in model_runs.items():

    print("\n\n################################################################")
    print("MODEL:", name)
    print("################################################################")

    # Convert labels
    if hasattr(y_test, "to_numpy"):
        y_true = y_test.to_numpy()
    else:
        y_true = np.asarray(y_test)

    # Clean predictions from the trained model
    y_pred = model.predict(X_test)

    # Basic clean metrics
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print("\nCLEAN PERFORMANCE")
    print("----------------------------")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # Row for summary table
    row = {
        "Model": name,
        "Clean Accuracy": acc,
        "Clean F1": f1
    }

    # ============================
    #   HOPSKIPJUMP RESULTS
    # ============================
    if "results_hsj" in globals() and name in results_hsj:
        hsj_clean = results_hsj[name]["clean_acc"]
        hsj_adv   = results_hsj[name]["adv_acc"]
        hsj_drop  = hsj_clean - hsj_adv

        print("\nHOPSKIPJUMP ADVERSARIAL RESULTS")
        print("----------------------------")
        print("Clean Accuracy (HSJ dict):", hsj_clean)
        print("Adv Accuracy   (HSJ):     ", hsj_adv)
        print("Accuracy Drop:", hsj_drop)

        row["HSJ Adv Accuracy"] = hsj_adv
        row["HSJ Drop"] = hsj_drop
    else:
        print("\nNo HSJ results recorded for this model in results_hsj.")

    hsj_summary_rows.append(row)

# Create dataframe summary
hsj_metrics_summary_df = pd.DataFrame(hsj_summary_rows)
print("\n\n===================== HSJ SUMMARY DATAFRAME =====================\n")
display(hsj_metrics_summary_df)

hsj_metrics_summary_df




===================== HOPSKIPJUMP METRICS SUMMARY =====================



################################################################
MODEL: LogisticRegression
################################################################

CLEAN PERFORMANCE
----------------------------
Accuracy: 0.9164214243672749
Precision: 0.9191983360683207
Recall: 0.9164214243672749
F1 Score: 0.9175247607419302

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1352
           1       0.77      0.83      0.80       347

    accuracy                           0.92      1699
   macro avg       0.87      0.89      0.87      1699
weighted avg       0.92      0.92      0.92      1699

Confusion Matrix:
[[1268   84]
 [  58  289]]

No HSJ results recorded for this model in results_hsj.


################################################################
MODEL: NeuralNet
################################################################

CL

,Model,Clean Accuracy,Clean F1
0,LogisticRegression,0.916421,0.917525
1,NeuralNet,0.945851,0.945674
2,RandomForest,0.945262,0.943670
3,SVM,0.929880,0.925855
4,XGBoost,0.943496,0.942108


,Model,Clean Accuracy,Clean F1
0,LogisticRegression,0.916421,0.917525
1,NeuralNet,0.945851,0.945674
2,RandomForest,0.945262,0.943670
3,SVM,0.929880,0.925855
4,XGBoost,0.943496,0.942108


In [ ]:
from sklearn.linear_model import LogisticRegression
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump
import numpy as np


def evaluate_robustness_hsj_with_detector(name, model, X_test, y_test,
                                          max_samples=100, max_iter=10):
    """
    Full HopSkipJump evaluation:
       - Runs HSJ on subset of samples
       - Computes clean and adversarial accuracy
       - Builds binary detector dataset (clean=0, adv=1)
       - Trains LogisticRegression detector
       - Reports DetectorTPR, DetectorFPR
    """

    # convert X
    if hasattr(X_test, "to_numpy"):
        X = X_test.to_numpy().astype(np.float32)
    else:
        X = np.asarray(X_test, dtype=np.float32)

    # convert y
    if hasattr(y_test, "to_numpy"):
        y_raw = y_test.to_numpy()
    else:
        y_raw = np.asarray(y_test)

    # label mapping
    classes, y_int = np.unique(y_raw, return_inverse=True)

    n = min(max_samples, len(X))
    if n == 0:
        return {
            "Model": name,
            "DetectorTPR": 0.0,
            "DetectorFPR": 0.0,
            "CleanAccSubset": 0.0,
            "AdvAccSubset": 0.0,
            "AccDropSubset": 0.0,
            "NumAttacked": 0,
        }

    X = X[:n]
    y = y_int[:n]

    # clip values based on data
    clip_values = (float(X.min()), float(X.max()))

    # ART classifier
    art_clf = SklearnClassifier(model=model, clip_values=clip_values)

    # HopSkipJump attack
    hsj = HopSkipJump(
        classifier=art_clf,
        max_iter=max_iter,
        max_eval=10000,
        init_eval=100,
        init_size=1,
        targeted=False,
    )

    # storage for detector
    det_X = []
    det_y = []

    clean_correct = 0
    adv_correct = 0

    for i in range(n):
        xi = X[i:i+1]
        yi = np.array([y[i]])

        pred_clean = np.argmax(art_clf.predict(xi), axis=1)

        # add clean to detector dataset
        det_X.append(xi[0])
        det_y.append(0)

        # adversarial generation
        x_adv = hsj.generate(x=xi, y=yi)
        pred_adv = np.argmax(art_clf.predict(x_adv), axis=1)

        # add adversarial sample to detector dataset
        det_X.append(x_adv[0])
        det_y.append(1)

        clean_correct += int(pred_clean[0] == yi[0])
        adv_correct   += int(pred_adv[0] == yi[0])

    clean_acc = clean_correct / n
    adv_acc   = adv_correct   / n
    acc_drop  = clean_acc - adv_acc

    det_X = np.vstack(det_X).astype(np.float32)
    det_y = np.asarray(det_y, dtype=int)

    # train detector
    det = LogisticRegression(max_iter=200)
    det.fit(det_X, det_y)

    det_preds = det.predict(det_X)

    # detector metrics
    tp = np.sum((det_preds == 1) & (det_y == 1))
    fn = np.sum((det_preds == 0) & (det_y == 1))
    fp = np.sum((det_preds == 1) & (det_y == 0))
    tn = np.sum((det_preds == 0) & (det_y == 0))

    det_tpr = tp / (tp + fn + 1e-9)
    det_fpr = fp / (fp + tn + 1e-9)

    return {
        "Model": name,
        "DetectorTPR": float(det_tpr),
        "DetectorFPR": float(det_fpr),
        "CleanAccSubset": float(clean_acc),
        "AdvAccSubset": float(adv_acc),
        "AccDropSubset": float(acc_drop),
        "NumAttacked": int(n),
    }


In [ ]:
results = []

for name, (model, X_test, y_test) in model_runs.items():

    # Skip all XGBoost models for HSJ
    lname = name.lower()
    if "xgb" in lname or "xgboost" in lname:
        print(f"Skipping {name} (HSJ does not support XGBoost reliably)")
        continue

    print(f"\n=== HSJ Robustness for {name} ===")

    res = evaluate_robustness_hsj_with_detector(
        name=name,
        model=model,
        X_test=X_test,
        y_test=y_test,
        max_samples=100
    )

    print(res)
    results.append(res)

df_hsj = pd.DataFrame(results)
df_hsj



=== HSJ Robustness for LogisticRegression ===


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 999.60it/s]
C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'Model': 'LogisticRegression', 'DetectorTPR': 0.599999999994, 'DetectorFPR': 0.5899999999941, 'CleanAccSubset': 0.93, 'AdvAccSubset': 0.79, 'AccDropSubset': 0.14, 'NumAttacked': 100}

=== HSJ Robustness for NeuralNet ===


HopSkipJump: 100%|██████████| 1/1 [00:00<?, ?it/s]
C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'Model': 'NeuralNet', 'DetectorTPR': 0.5399999999946, 'DetectorFPR': 0.5399999999946, 'CleanAccSubset': 0.96, 'AdvAccSubset': 0.8, 'AccDropSubset': 0.15999999999999992, 'NumAttacked': 100}

=== HSJ Robustness for RandomForest ===


HopSkipJump:   0%|          | 0/1 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# ==========================================
# BINARYINPUTDETECTOR SUMMARY (SKLEARN + TORCH)
# ==========================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression   # only for your main models if needed
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump
from art.defences.detector.evasion import BinaryInputDetector

# NEW: PyTorch backend for the detector
import torch
import torch.nn as nn
import torch.optim as optim
from art.estimators.classification import PyTorchClassifier


# Reuse your HSJ params if they exist
_default_hsj_kwargs = dict(
    max_iter=20,
    max_eval=10000,
    init_eval=100,
    init_size=10,
    targeted=False,
    norm=2,
)
hsj_params = _default_hsj_kwargs.copy()
if "hsj_kwargs" in globals():
    hsj_params.update(hsj_kwargs)


def _predict_labels_art(art_clf, x):
    """Safe prediction wrapper: handles 1D or 2D outputs."""
    preds = np.asarray(art_clf.predict(x))
    if preds.ndim == 1:               # already labels
        return preds.astype(int)
    return np.argmax(preds, axis=1)    # probability matrix


class DetectorNet(nn.Module):
    """Tiny MLP detector: input -> hidden -> hidden -> 2-class output."""
    def __init__(self, input_dim, hidden1=32, hidden2=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 2)   # 2 classes: 0=clean, 1=adv
        )

    def forward(self, x):
        return self.net(x)


def build_torch_detector(input_dim, clip_values):
    """Build an ART PyTorchClassifier suitable for BinaryInputDetector."""
    model = DetectorNet(input_dim)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    art_clf = PyTorchClassifier(
        model=model,
        loss=loss_fn,
        optimizer=optimizer,
        input_shape=(input_dim,),
        nb_classes=2,
        clip_values=clip_values,
        device_type="cpu",  # change to "gpu" if you have CUDA
    )
    return art_clf


def run_hsj_with_binary_detector(model_runs, max_samples=50):
    rows = []

    for name, (model, X_test, y_test) in model_runs.items():
        print(f"\n=== {name}: HSJ + BinaryInputDetector ===")

        # Convert data
        X = X_test.to_numpy().astype(np.float32) if hasattr(X_test, "to_numpy") else np.asarray(X_test, dtype=np.float32)
        y_raw = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)

        # Encode labels 0..K-1
        classes, y_int = np.unique(y_raw, return_inverse=True)

        n = min(max_samples, len(X))
        if n == 0:
            print(f"{name}: no samples, skipping")
            continue

        X = X[:n]
        y = y_int[:n]

        clip_values = (float(X.min()), float(X.max()))

        # Wrap your existing sklearn model for HSJ
        art_clf = SklearnClassifier(model=model, clip_values=clip_values)
        hsj = HopSkipJump(classifier=art_clf, **hsj_params)

        clean_correct = 0
        adv_correct = 0
        X_clean_list = []
        X_adv_list = []

        for i in range(n):
            xi = X[i:i+1]
            yi = np.array([y[i]])

            pred_clean = _predict_labels_art(art_clf, xi)
            x_adv = hsj.generate(x=xi, y=yi)
            pred_adv = _predict_labels_art(art_clf, x_adv)

            clean_correct += int(pred_clean[0] == yi[0])
            adv_correct += int(pred_adv[0] == yi[0])

            X_clean_list.append(xi)
            X_adv_list.append(x_adv)

        clean_acc = clean_correct / n
        adv_acc = adv_correct / n
        acc_drop = clean_acc - adv_acc

        # Build dataset for detector: clean = 0, adv = 1
        X_clean_det = np.vstack(X_clean_list)
        X_adv_det = np.vstack(X_adv_list)
        X_det = np.vstack([X_clean_det, X_adv_det])
        y_det = np.concatenate([
            np.zeros(len(X_clean_det), dtype=np.int64),
            np.ones(len(X_adv_det), dtype=np.int64),
        ])

        det_clip = (float(X_det.min()), float(X_det.max()))
        input_dim = X_det.shape[1]

        # PyTorch-based detector; this is what BinaryInputDetector expects
        det_art = build_torch_detector(input_dim=input_dim, clip_values=det_clip)

        detector = BinaryInputDetector(detector=det_art)
        detector.fit(X_det, y_det, nb_epochs=5, batch_size=32)

        # Evaluate detector
        _, det_clean = detector.detect(X_clean_det)   # True = flagged adversarial
        _, det_adv   = detector.detect(X_adv_det)

        det_tpr = float(det_adv.mean())    # correctly flags adversarial
        det_fpr = float(det_clean.mean())  # incorrectly flags clean

        print(f"{name}: clean_acc={clean_acc:.4f}, adv_acc={adv_acc:.4f}, drop={acc_drop:.4f}")
        print(f"{name}: detector TPR={det_tpr:.4f}, FPR={det_fpr:.4f}")

        rows.append({
            "Model": name,
            "DetectorTPR": det_tpr,
            "DetectorFPR": det_fpr,
            "CleanAccSubset": clean_acc,
            "AdvAccSubset": adv_acc,
            "AccDropSubset": acc_drop,
            "NumAttacked": n,
        })

    return pd.DataFrame(rows)


binary_detector_summary_df = run_hsj_with_binary_detector(model_runs, max_samples=50)
display(binary_detector_summary_df)
binary_detector_summary_df



=== LogisticRegression: HSJ + MANUAL DETECTOR ===


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 333.46it/s]


LogisticRegression: Detector TPR=0.5800, FPR=0.5800

=== NeuralNet: HSJ + MANUAL DETECTOR ===


HopSkipJump: 100%|██████████| 1/1 [00:00<00:00, 333.52it/s]


NeuralNet: Detector TPR=0.4000, FPR=0.4000

=== RandomForest: HSJ + MANUAL DETECTOR ===


HopSkipJump:   0%|          | 0/1 [00:01<?, ?it/s]


KeyboardInterrupt: 